In [5]:
def parse_arguments():
    parser = argparse.ArgumentParser(
        description="Impor dataset Retail Rocket (events.csv) ke database Neo4J."
    )
    parser.add_argument(
        "-u", "-user", default="neo4j", help="Username Neo4j (default: neo4j)"
    )
    parser.add_argument(
        "-p",
        "--password",
        default="neo4j",
        help="Password Neo4J (default: neo4j)",
    )
    parser.add_argument(
        "-b",
        "--bolt",
        default="neo4j://127.0.0.1:7687",
        help="Neo4J URI / Bolt (default: neo4j://127.0.0.1:7687)",
    )
    parser.add_argument(
        "-s",
        "--source",
        default="data",
        help="Direktori/folder sumber data (default: data)",
    )
    return parser.parse_args()

In [12]:
from types import SimpleNamespace

args = SimpleNamespace(
    source=".",
    bolt="bolt://localhost:7687",
    user="neo4j",
    password="12345678"
)

csv_file_path = os.path.join(args.source, "archive\\events.csv") 

if not os.path.exists(csv_file_path): 
    print(f"Error: File '{csv_file_path}' tidak ditemukan!") 
    sys.exit(1) 

In [13]:
try: 
    driver = GraphDatabase.driver( 
        args.bolt, auth=(args.user, args.password) 
    ) 
    driver.verify_connectivity() 
except Exception as e: 
    print(f"Gagal terhubung ke database: {e}") 
    sys.exit(1) 

In [16]:
def create_constraints(session):
    session.run("""
        CREATE CONSTRAINT transaction_id_unique IF NOT EXISTS
        FOR (t:Transaction)
        REQUIRE t.transactionid IS UNIQUE
    """)

In [20]:
def import_batch(tx, records):
    query = """
    UNWIND $records AS row
    MERGE (t:Transaction {transactionid: row.transactionid})
    SET t += row
    """
    tx.run(query, records=records)

In [21]:
import pandas as pd
from tqdm.auto import tqdm

batch_size = 1000

with driver.session() as session:
    create_constraints(session)

    print(f"Membaca file data dari: {csv_file_path}")

    chunks = pd.read_csv(
        csv_file_path,
        chunksize=batch_size
    )

    # Menghitung total baris dan batch
    total_rows = sum(
        len(chunk) for chunk in pd.read_csv(
            csv_file_path,
            chunksize=batch_size
        )
    )
    total_chunks = (total_rows + batch_size - 1) // batch_size

    print("Memulai proses impor data ke Neo4j...")

    for chunk in tqdm(
        chunks,
        total=total_chunks,
        desc="Mengimpor Batch Data"
    ):
        # Membersihkan data null/NaN pada transactionid
        chunk["transactionid"] = (
            chunk["transactionid"].fillna("").astype(str)
        )

        records = chunk.to_dict(orient="records")
        session.execute_write(import_batch, records)

driver.close()
print("Proses impor berhasil.")

Membaca file data dari: .\archive\events.csv
Memulai proses impor data ke Neo4j...


Mengimpor Batch Data:   0%|          | 0/2757 [00:00<?, ?it/s]

Proses impor berhasil.


In [22]:
def import_batch(tx, batch): 
    cypher_query = """ 
    UNWIND $batch AS row 
    MERGE (v:Visitor {visitorId: row.visitorid}) 
    MERGE (i:Item {itemId: row.itemid}) 
    WITH v, i, row
    CALL { 
        WITH v, i, row 
        WITH v, i, row WHERE row.event = 'view' 
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i) 
        UNION ALL 
        WITH v, i, row 
        WITH v, i, row WHERE row.event = 'addtocart' 
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i) 
        UNION ALL 
        WITH v, i, row 
        WITH v, i, row WHERE row.event = 'transaction' 
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i) 
    } 
    """ 
    tx.run(cypher_query, batch=batch) 